In [1]:
# import pyomo.environ as pyo
# from pyomo.dae import ContinuousSet, DerivativeVar, Simulator
# import numpy as np
# from pathlib import Path
# from metrics_logger import log_benchmark
# from pyomo.contrib.parmest.experiment import Experiment
# ## This code simulates xdot = -ax + bsin(wt) + u , a,b are the unknown parameter that are to be estimated. 
# ##The parameter w is known and set to 1.0


# # ========================
# class TwoParameterExperiment(Experiment):
#     def __init__(self, data, nfe, ncp):
#         """
#         Arguments
#         ---------
#         data: object containing vital experimental information
#         nfe: number of finite elements
#         ncp: number of collocation points for the finite elements
#         """
#         self.data = data
#         self.nfe = nfe
#         self.ncp = ncp
#         self.model = None

#         #############################
#         # End constructor definition

#     def get_labeled_model(self):
#         if self.model is None:
#             self.create_model()
#             self.finalize_model()
#             self.label_experiment()
#         return self.model

#     # Create flexible model without data
#     def create_model(self):
#         """
#         This is an example user model provided to DoE library.
#         It is a dynamic problem solved by Pyomo.DAE.

#         Return
#         ------
#         m: a Pyomo.DAE model
#         """

#         m = self.model = pyo.ConcreteModel()

#         # Model parameters
#         m.w = pyo.Param(mutable=False, initialize=1.0)

#         # Define model variables
#         ########################
#         # time
#         m.t = ContinuousSet(bounds=[0, 1])

#         # State and input
#         m.x = pyo.Var(m.t, within=pyo.Reals)
#         m.u = pyo.Var(m.t, within=pyo.Reals)

#         # Unknown parameter
#         m.a = pyo.Var(within=pyo.Reals)
#         m.b = pyo.Var(within=pyo.Reals)

#         # Differential variables wrt x
#         m.dxdt = DerivativeVar(m.x, wrt=m.t)

#         ########################
#         # End variable def.

#         # Equation definition
#         ########################

#         # Expression for the state evolution

#         # State odes
#         @m.Constraint(m.t)
#         def x_ode(m, t):
#             return m.dxdt[t] == m.a * m.x[t] + m.b * pyo.sin(m.w * m.x[t]) + m.u[t]


#         ########################
#         # End equation definition

#     def finalize_model(self):
#         """
#         Example finalize model function. There are two main tasks
#         here:

#             1. Extracting useful information for the model to align
#                with the experiment. (Here: CA0, t_final, t_control)
#             2. Discretizing the model subject to this information.

#         """
#         m = self.model

#         # Unpacking data before simulation
#         control_points = self.data["control_points"]

#         # Set initial state values for the experiment
#         m.x[0].value = self.data["x0"]

#         # Update model time `t` with time range and control time points
#         m.t.update(self.data["t_range"])
#         m.t.update(control_points)

#         # Fix the unknown parameter values
#         m.a.fix(self.data["a"])
#         m.b.fix(self.data["b"])

#         # Add upper and lower bounds to the design variable, CA[0]
#         m.x[0].setlb(self.data["x_bounds"][0])
#         m.x[0].setub(self.data["x_bounds"][1])

#         m.t_control = control_points

#         Discretizing the model
#         discr = pyo.TransformationFactory("dae.collocation")
#         discr.apply_to(m, nfe=self.nfe, ncp=self.ncp, wrt=m.t)

#         # Initializing control input in the model
#         cv = None
#         for t in m.t:
#             if t in control_points:
#                 cv = control_points[t]
#                 m.u[t].fix()
#             m.u[t].setlb(self.data["u_bounds"][0])
#             m.u[t].setub(self.data["u_bounds"][1])
#             m.u[t] = cv

#         # Make a constraint that holds control input constant between control time points
#         @m.Constraint(m.t - control_points)
#         def u_control(m, t):
#             """
#             Piecewise constant control input between control points
#             """
#             neighbour_t = max(tc for tc in control_points if tc < t)
#             return m.u[t] == m.u[neighbour_t]

#         #########################
#         # End model finalization

#     def label_experiment(self):
#         """
#         Example for annotating (labeling) the model with a
#         full experiment.
#         """
#         m = self.model

#         # Set measurement labels
#         m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         # Add CA to experiment outputs
#         m.experiment_outputs.update((m.x[t], None) for t in m.t_control)


#         # Adding error for measurement values (assuming no covariance and constant error for all measurements)
#         m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         meas_error = 1e-2  # Error in state measurement
#         # Add measurement error for CA
#         m.measurement_error.update((m.x[t], meas_error) for t in m.t_control)
   

#         # Identify design variables (experiment inputs) for the model
#         m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         # Add experimental input label for initial state
#         m.experiment_inputs[m.x[m.t.first()]] = None
#         # Add experimental input label for control input
#         m.experiment_inputs.update((m.u[t], None) for t in m.t_control)

#         # Add unknown parameter labels
#         m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         # Add labels to all unknown parameters with nominal value as the value
#         m.unknown_parameters.update((k, pyo.value(k)) for k in [m.a])

#         #########################
#         # End model labeling

In [2]:
# from pyomo.common.dependencies import numpy as np

# from pyomo.contrib.doe import DesignOfExperiments

# import pyomo.environ as pyo
# from pyomo.environ import (
#     ConcreteModel,
#     Var,
#     Param,
#     Constraint,
#     TransformationFactory,
#     SolverFactory,
#     Objective,
#     minimize,
#     value as pyovalue,
#     Suffix,
#     Expression,
#     sin,
#     NonNegativeReals,
# )



# data_ex = {"x0": 0.5, "x_bounds": [0.5, 1.5], "t_range": [0, 1],
#            "control_points": {"0": -1, "0.125": 1, "0.25": 1, "0.375": 1, "0.5": 1, "0.625": 1, "0.75": 1,
#                               "0.875": 1, "1": 1}, "u_bounds": [-1.0, 1.0], "a": -0.1, "b":1.2}
# # Put control input control time points into correct format for two parameter experiment
# data_ex["control_points"] = {
#     float(k): v for k, v in data_ex["control_points"].items()
# }

# # Create a two parameterExperiment object; data and discretization information are part
# # of the constructor of this object
# experiment = TwoParameterExperiment(data=data_ex, nfe=10, ncp=3)

# # Use a central difference, with step size 1e-3
# fd_formula = "central"
# step_size = 1e-3

# # Use the determinant objective with scaled sensitivity matrix
# objective_option = "determinant"
# scale_nominal_param_value = True

# # Create the DesignOfExperiments object
# # We will not be passing any prior information in this example.
# # We also will rely on the initialization routine within
# # the DesignOfExperiments class.
# doe_obj = DesignOfExperiments(
#     experiment,
#     fd_formula=fd_formula,
#     step=step_size,
#     objective_option=objective_option,
#     scale_constant_value=1,
#     scale_nominal_param_value=scale_nominal_param_value,
#     prior_FIM=None,
#     jac_initial=None,
#     fim_initial=None,
#     L_diagonal_lower_bound=1e-7,
#     solver=SolverFactory('IPOPT'),
#     tee=False,
#     get_labeled_model_args=None,
#     _Cholesky_option=True,
#     _only_compute_fim_lower=True,
# )

# # # Make design ranges to compute the full factorial design
# # design_ranges = {"x[0]": [0.5, 1.5, 20], "u[0]": [-1, 1, 20\]}

# # # Compute the full factorial design with the sequential FIM calculation
# # doe_obj.compute_FIM_full_factorial(design_ranges=design_ranges, method="sequential")

# # # Plot the results
# # doe_obj.draw_factorial_figure(
# #     sensitivity_design_variables=["x[0]", "u[0]"],
# #     fixed_design_variables={
# #         "u[0.125]": 1,
# #         "u[0.25]": 1,
# #         "u[0.375]": 1,
# #         "u[0.5]": 1,
# #         "u[0.625]": 1,
# #         "u[0.75]": 1,
# #         "u[0.875]": 1,
# #         "u[1]": 1,
# #     },
# #     title_text="Single State Example",
# #     xlabel_text="State",
# #     ylabel_text="Input",
# #     figure_file_name="example_one_state_compute_FIM",
# #     log_scale=False,
# # )


# doe_obj.run_doe()

# # Print out a results summary
# print("Optimal experiment values: ")
# print(
#     "\tInitial state: {:.2f}".format(
#         doe_obj.results["Experiment Design"][0]
#     )
# )
# print(
#     ("\t Control input values: [" + "{:.2f}, " * 8 + "{:.2f}]").format(
#         *doe_obj.results["Experiment Design"][1:]
#     )
# )
# print("FIM at optimal design:\n {}".format(np.array(doe_obj.results["FIM"])))
# print(
#     "Objective value at optimal design: {:.2f}".format(
#         pyo.value(doe_obj.model.objective)
#     )
# )

# print(doe_obj.results["Experiment Design Names"])

# #print(sorted(doe_obj.results.keys()))

# print("Solve time (s):", doe_obj.results["Solve Time"])
# print("Build time (s):", doe_obj.results["Build Time"])
# print("Initialization time (s):", doe_obj.results["Initialization Time"])
# print("Total wall time (s):", doe_obj.results["Wall-clock Time"])

# ###################
# # End optimal DoE



In [3]:
# from metrics_logger import log_benchmark

# NOTEBOOK_ID = "2"
# scenario = "central"
# log_benchmark(
#     notebook_id = NOTEBOOK_ID,
#     scenario = scenario,
#     solve_time_s = doe_obj.results["Solve Time"],
#     build_time_s = doe_obj.results["Build Time"],
#     init_time_s = doe_obj.results["Initialization Time"],
#     total_wall_time_s = doe_obj.results["Wall-clock Time"]
# )

In [4]:
#  ___________________________________________________________________________
#
#  Pyomo: Python Optimization Modeling Objects
#  Copyright (c) 2008-2025
#  National Technology and Engineering Solutions of Sandia, LLC
#  Under the terms of Contract DE-NA0003525 with National Technology and
#  Engineering Solutions of Sandia, LLC, the U.S. Government retains certain
#  rights in this software.
#  This software is distributed under the 3-clause BSD License.
#  ___________________________________________________________________________

# === Minimal deps ===
# One-time in your env: conda install -y -c conda-forge openpyxl

from pathlib import Path
import shutil

import pyomo.environ as pyo
from pyomo.dae import ContinuousSet, DerivativeVar
from pyomo.contrib.parmest.experiment import Experiment
from pyomo.contrib.doe import DesignOfExperiments
import numpy as np

# --------------------
# IPOPT setup (avoid CRC “Solver (IPOPT) not available” & MA27 memory issues)
# --------------------
IPOPT_BIN = shutil.which("ipopt")
assert IPOPT_BIN, "ipopt executable not found on PATH (activate your env)."

def make_ipopt():
    s = pyo.SolverFactory("ipopt", executable=IPOPT_BIN)
    s.options["linear_solver"] = "mumps"   # avoid MA27 iflag=-4
    return s

# --------------------
# Excel helpers (same metrics.xlsx as before)
# --------------------
from openpyxl import Workbook, load_workbook

def _ensure_book_and_sheet(xlsx_path: Path, sheet_name: str,
                           header_cols: list[str], notebook_id: str, scenario: str):
    """
    Ensure outputs/metrics.xlsx exists and has a sheet `sheet_name`.
    On first creation: write NOTEBOOK_ID & scenario once, blank line, then header row.
    """
    xlsx_path.parent.mkdir(parents=True, exist_ok=True)
    if xlsx_path.exists():
        wb = load_workbook(xlsx_path)
    else:
        wb = Workbook()
        if wb.sheetnames == ["Sheet"]:
            wb.remove(wb["Sheet"])

    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        if ws.max_row == 1 and all(c.value is None for c in ws[1]):
            ws.append(["NOTEBOOK_ID", notebook_id, "SCENARIO", scenario])
            ws.append([])
            ws.append(header_cols)
    else:
        ws = wb.create_sheet(title=sheet_name[:31])
        ws.append(["NOTEBOOK_ID", notebook_id, "SCENARIO", scenario])
        ws.append([])
        ws.append(header_cols)

    return wb, ws

def _append_row(ws, row): ws.append(row)
def _save_wb(wb, xlsx_path: Path): wb.save(xlsx_path)

# ========================
# Two-parameter ODE experiment
# ========================
class TwoParameterExperiment(Experiment):
    def __init__(self, data, nfe, ncp):
        self.data = data
        self.nfe = nfe
        self.ncp = ncp
        self.model = None

    def get_labeled_model(self):
        if self.model is None:
            self.create_model()
            self.finalize_model()
            self.label_experiment()
        return self.model

    def create_model(self):
        m = self.model = pyo.ConcreteModel()

        # Parameter
        m.w = pyo.Param(mutable=False, initialize=1.0)

        # Time
        m.t = ContinuousSet(bounds=[0, 1])

        # State & input
        m.x = pyo.Var(m.t, within=pyo.Reals)
        m.u = pyo.Var(m.t, within=pyo.Reals)

        # Unknown parameters
        m.a = pyo.Var(within=pyo.Reals)
        m.b = pyo.Var(within=pyo.Reals)

        # Derivative
        m.dxdt = DerivativeVar(m.x, wrt=m.t)

        # Dynamics: ẋ = a x + b sin(w x) + u
        @m.Constraint(m.t)
        def x_ode(m, t):
            return m.dxdt[t] == m.a * m.x[t] + m.b * pyo.sin(m.w * m.x[t]) + m.u[t]

    def finalize_model(self):
        m = self.model
        control_points = self.data["control_points"]

        # IC
        m.x[0].value = self.data["x0"]

        # Time grid & control points
        m.t.update(self.data["t_range"])
        m.t.update(control_points)

        # Fix unknowns at nominal
        m.a.fix(self.data["a"])
        m.b.fix(self.data["b"])

        # Bounds on design variable x(0)
        m.x[0].setlb(self.data["x_bounds"][0])
        m.x[0].setub(self.data["x_bounds"][1])

        m.t_control = control_points

        # Discretize
        pyo.TransformationFactory("dae.collocation").apply_to(m, nfe=self.nfe, ncp=self.ncp, wrt=m.t)

        # Initialize u(t) piecewise-constant
        cv = None
        for t in m.t:
            if t in control_points:
                cv = control_points[t]
                m.u[t].fix()
            m.u[t].setlb(self.data["u_bounds"][0])
            m.u[t].setub(self.data["u_bounds"][1])
            m.u[t] = cv

        @m.Constraint(m.t - control_points)
        def u_control(m, t):
            neighbour_t = max(tc for tc in control_points if tc < t)
            return m.u[t] == m.u[neighbour_t]

    def label_experiment(self):
        m = self.model

        # Measured outputs: x at control times
        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_outputs.update((m.x[t], None) for t in m.t_control)

        # Measurement error
        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        meas_error = 1e-2
        m.measurement_error.update((m.x[t], meas_error) for t in m.t_control)

        # Design variables: x(0) and u at control times
        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_inputs[m.x[m.t.first()]] = None
        m.experiment_inputs.update((m.u[t], None) for t in m.t_control)

        # Unknowns (labels only; values fixed above)
        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.unknown_parameters.update((k, pyo.value(k)) for k in [m.a, m.b])

# --------------------
# Static data + options
# --------------------
data_ex = {
    "x0": 0.5,
    "x_bounds": [0.5, 1.5],
    "t_range": [0, 1],
    "control_points": {
        0.0: -1, 0.125: 1, 0.25: 1, 0.375: 1, 0.5: 1,
        0.625: 1, 0.75: 1, 0.875: 1, 1.0: 1
    },
    "u_bounds": [-1.0, 1.0],
    "a": -0.1,
    "b": 1.2
}

fd_formula = "central"
step_size = 1e-3
objective_option = "determinant"
scale_nominal_param_value = True

# --------------------
# Excel config (same workbook as before)
# --------------------
NOTEBOOK_ID = "2"
scenario = "central"
XLSX_PATH = Path("outputs/metrics.xlsx")
SHEET_NAME = f"{NOTEBOOK_ID}_{scenario}"[:31]
ROW_HEADER = ["run", "solve_time_s", "build_time_s", "init_time_s", "total_wall_time_s", "optimal_experiment_values"]

_wb, _ws = _ensure_book_and_sheet(XLSX_PATH, SHEET_NAME, ROW_HEADER, NOTEBOOK_ID, scenario)
_save_wb(_wb, XLSX_PATH)

# --------------------
# One run = build fresh DoE + call run_doe() once
# --------------------
def run_single():
    experiment = TwoParameterExperiment(data=data_ex, nfe=10, ncp=3)
    doe_obj = DesignOfExperiments(
        experiment,
        fd_formula=fd_formula,
        step=step_size,
        objective_option=objective_option,
        scale_constant_value=1,
        scale_nominal_param_value=scale_nominal_param_value,
        prior_FIM=None,
        jac_initial=None,
        fim_initial=None,
        L_diagonal_lower_bound=1e-7,
        solver=make_ipopt(),
        tee=False,
        get_labeled_model_args=None,
        _Cholesky_option=True,
        _only_compute_fim_lower=True,
    )
    doe_obj.run_doe()

    # metrics
    solve_time = doe_obj.results["Solve Time"]
    build_time = doe_obj.results["Build Time"]
    init_time  = doe_obj.results["Initialization Time"]
    total_time = doe_obj.results["Wall-clock Time"]

    # optimal design values (CSV string)
    design_vals = doe_obj.results["Experiment Design"]
    design_vals_str = ",".join(f"{v:.6g}" for v in design_vals)

    return solve_time, build_time, init_time, total_time, design_vals_str

# --------------------
# Run 100 times, append rows to the same Excel sheet
# --------------------
wb = load_workbook(XLSX_PATH) if XLSX_PATH.exists() else Workbook()
ws = wb[SHEET_NAME] if SHEET_NAME in wb.sheetnames else wb.create_sheet(title=SHEET_NAME)
if ws.max_row == 1 and all(c.value is None for c in ws[1]):
    ws.append(["NOTEBOOK_ID", NOTEBOOK_ID, "SCENARIO", scenario])
    ws.append([])
    ws.append(ROW_HEADER)

for run in range(1, 101):
    print(f"Run {run}")
    solve_time, build_time, init_time, total_time, design_vals_str = run_single()
    _append_row(ws, [run, solve_time, build_time, init_time, total_time, design_vals_str])

_save_wb(wb, XLSX_PATH)

# (Optional) quick summary print of the last run could be added if you want.


Run 1
Run 2
Run 3
Run 4
Run 5
Run 6
Run 7
Run 8
Run 9
Run 10
Run 11
Run 12
Run 13
Run 14
Run 15
Run 16
Run 17
Run 18
Run 19
Run 20
Run 21
Run 22
Run 23
Run 24
Run 25
Run 26
Run 27
Run 28
Run 29
Run 30
Run 31
Run 32
Run 33
Run 34
Run 35
Run 36
Run 37
Run 38
Run 39
Run 40
Run 41
Run 42
Run 43
Run 44
Run 45
Run 46
Run 47
Run 48
Run 49
Run 50
Run 51
Run 52
Run 53
Run 54
Run 55
Run 56
Run 57
Run 58
Run 59
Run 60
Run 61
Run 62
Run 63
Run 64
Run 65
Run 66
Run 67
Run 68
Run 69
Run 70
Run 71
Run 72
Run 73
Run 74
Run 75
Run 76
Run 77
Run 78
Run 79
Run 80
Run 81
Run 82
Run 83
Run 84
Run 85
Run 86
Run 87
Run 88
Run 89
Run 90
Run 91
Run 92
Run 93
Run 94
Run 95
Run 96
Run 97
Run 98
Run 99
Run 100
